# Lab 1 — Model Comparison

Side-by-side evaluation of all four sentiment classifiers — **ANN**, **BiLSTM**, **DistilBERT**, and **RoBERTa** — across three datasets of increasing scale and complexity.

| Dataset | Task | Test samples |
|---------|------|--------------|
| 1K | Binary sentiment (Negative / Positive) | 100 |
| 25K | Binary sentiment (Negative / Positive) | 2 500 |
| Video Games (VG) | 5-class star rating (1–5) | ~256 K |

Each dataset section shows per-model classification reports, a comparison table (Accuracy, Macro F1, Weighted F1), and confusion matrices. An overall summary table at the end collects all 12 results.

## Setup & Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from data_loading_code import preprocess_pandas
from transformer_utils import (
    build_tf_datasets,
    compute_metrics_tf,
    evaluate_tf,
    plot_confusion_matrix_tf,
)
from utils import (
    device_check,
    validate,
    plot_confusion_matrix,
    load_ann_run,
    load_bilstm_run,
)

device = device_check()

In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent
SPLITS_DIR   = PROJECT_DIR / 'data' / 'splits'
MODELS_DIR   = PROJECT_DIR / 'models'
MAX_LENGTH   = 128

DISTILBERT_NAME = 'distilbert-base-uncased'
ROBERTA_NAME    = 'roberta-base'

tokenizer_distilbert = AutoTokenizer.from_pretrained(DISTILBERT_NAME)
tokenizer_roberta    = AutoTokenizer.from_pretrained(ROBERTA_NAME)

PyTorch: 2.11.0+cpu | Python: 3.13.7 | OS: Windows 11
CUDA available: False
Using cpu


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

c:\Users\Oscar\Projects\D7047E\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Oscar\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
test_1k  = pd.read_csv(SPLITS_DIR / '1k_test.csv')
test_25k = pd.read_csv(SPLITS_DIR / '25k_test.csv')
test_vg  = pd.read_csv(SPLITS_DIR / 'vg_test.csv')
# Remap VG labels 1-5 -> 0-4
test_vg['Class'] = test_vg['Class'].astype(int) - 1

## Helper functions

Preprocessing, encoding, and evaluation wrappers so each dataset section stays concise.

In [ ]:
def preprocess_seq(df: pd.DataFrame) -> pd.DataFrame:
    """Lightweight text cleaning for the BiLSTM (keeps stopwords)."""
    df = df.copy()
    s = df['Sentence'].str.lower()
    s = s.str.replace(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', regex=True)
    s = s.str.replace(r'(?:\d{1,3}\.){3}\d{1,3}', '', regex=True)
    s = s.str.replace(r'[^\w\s]', ' ', regex=True)
    s = s.str.replace(r'\d+', ' ', regex=True)
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()
    df['Sentence'] = s
    return df


def encode_corpus(texts, vocab, max_len):
    return torch.tensor(
        [vocab.encode(t, max_len) for t in texts],
        dtype=torch.long,
    )


def eval_ann(model, vectorizer, svd, test_df, batch_size=512):
    """Preprocess â†’ TF-IDF â†’ SVD â†’ DataLoader â†’ validate. Returns (y_true, y_pred)."""
    test_proc = preprocess_pandas(test_df.copy())
    X_tfidf   = vectorizer.transform(test_proc['Sentence'])
    X_svd     = svd.transform(X_tfidf)
    X_t       = torch.tensor(X_svd, dtype=torch.float32)
    y_t       = torch.tensor(test_df['Class'].values, dtype=torch.long)
    loader    = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=False)
    _, _, y_true, y_pred = validate(model, loader, nn.CrossEntropyLoss())
    return y_true, y_pred


def eval_bilstm(model, vocab, max_seq_len, test_df, batch_size=512):
    """Preprocess â†’ encode â†’ DataLoader â†’ validate. Returns (y_true, y_pred)."""
    test_proc = preprocess_seq(test_df.copy())
    X_t       = encode_corpus(test_proc['Sentence'].tolist(), vocab, max_seq_len)
    y_t       = torch.tensor(test_df['Class'].values, dtype=torch.long)
    loader    = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=False)
    _, _, y_true, y_pred = validate(model, loader, nn.CrossEntropyLoss())
    return y_true, y_pred


def report(label, y_true, y_pred, class_names):
    """Print a sklearn classification report."""
    print(f"Classification Report: {label}\n")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))


def compute_scores(name, y_true, y_pred):
    return {
        'Model':       name,
        'Accuracy':    accuracy_score(y_true, y_pred),
        'Macro F1':    f1_score(y_true, y_pred, average='macro',    zero_division=0),
        'Weighted F1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }


def get_trainer(model):
    args = TrainingArguments(
        output_dir='tmp_eval',
        per_device_eval_batch_size=64,
        dataloader_num_workers=4,
        report_to='none',
    )
    return Trainer(model=model, args=args, compute_metrics=compute_metrics_tf)


def show_cm_grid(specs, ncols=2, cell_size=5):
    """Display confusion matrices in a grid without modifying plot_confusion_matrix_tf.

    specs : list of tuples passed straight to plot_confusion_matrix_tf:
            (y_true, y_pred, num_classes, class_names, title[, normalize])
    """
    import io
    import matplotlib.image as mpimg

    original_show = plt.show
    imgs = []

    def _capture():
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
        buf.seek(0)
        imgs.append(mpimg.imread(buf))
        plt.close()

    plt.show = _capture
    try:
        for spec in specs:
            plot_confusion_matrix_tf(*spec)
    finally:
        plt.show = original_show

    import matplotlib.pyplot as plt_grid
    nrows = (len(imgs) + ncols - 1) // ncols
    fig, axes = plt_grid.subplots(nrows, ncols, figsize=(cell_size * ncols, cell_size * nrows))
    axes_flat = np.array(axes).flat
    for ax, img in zip(axes_flat, imgs):
        ax.imshow(img)
        ax.axis('off')
    for ax in list(axes_flat)[len(imgs):]:
        ax.set_visible(False)
    plt_grid.tight_layout()
    plt_grid.show()


## Load all models

In [ ]:
# ===== ANN =====
ann_model_1k,  ann_vec_1k,  ann_svd_1k  = load_ann_run(MODELS_DIR / 'ann_1k',  device)
ann_model_25k, ann_vec_25k, ann_svd_25k = load_ann_run(MODELS_DIR / 'ann_25k', device)
ann_model_vg,  ann_vec_vg,  ann_svd_vg  = load_ann_run(MODELS_DIR / 'ann_vg',  device)

In [ ]:
# ===== BiLSTM =====
bilstm_model_1k,  bilstm_vocab_1k,  bilstm_maxlen_1k  = load_bilstm_run(MODELS_DIR / 'bilstm_1k',  device)
bilstm_model_25k, bilstm_vocab_25k, bilstm_maxlen_25k = load_bilstm_run(MODELS_DIR / 'bilstm_25k', device)
bilstm_model_vg,  bilstm_vocab_vg,  bilstm_maxlen_vg  = load_bilstm_run(MODELS_DIR / 'bilstm_vg',  device)

In [ ]:
# ===== DistilBERT =====
distilbert_1k  = AutoModelForSequenceClassification.from_pretrained(MODELS_DIR / 'distilbert_1k').to(device)
distilbert_25k = AutoModelForSequenceClassification.from_pretrained(MODELS_DIR / 'distilbert_25k').to(device)
distilbert_vg  = AutoModelForSequenceClassification.from_pretrained(MODELS_DIR / 'distilbert_vg').to(device)

In [ ]:
# ===== RoBERTa =====
roberta_1k  = AutoModelForSequenceClassification.from_pretrained(MODELS_DIR / 'roberta_1k').to(device)
roberta_25k = AutoModelForSequenceClassification.from_pretrained(MODELS_DIR / 'roberta_25k').to(device)
roberta_vg  = AutoModelForSequenceClassification.from_pretrained(MODELS_DIR / 'roberta_vg').to(device)

## Build transformer test datasets

One tokenized dataset per (model family, split). The same test DataFrame is passed
for all three slots â€” only the `'test'` key is used.

In [ ]:
def build_test_ds(test_df, tokenizer):
    return build_tf_datasets(
        train_df=test_df, val_df=test_df, test_df=test_df,
        tokenizer=tokenizer, text_col='Sentence', label_col='Class',
        max_length=MAX_LENGTH,
    )['test']

ds_1k_distil   = build_test_ds(test_1k,  tokenizer_distilbert)
ds_25k_distil  = build_test_ds(test_25k, tokenizer_distilbert)
ds_vg_distil   = build_test_ds(test_vg,  tokenizer_distilbert)

ds_1k_roberta  = build_test_ds(test_1k,  tokenizer_roberta)
ds_25k_roberta = build_test_ds(test_25k, tokenizer_roberta)
ds_vg_roberta  = build_test_ds(test_vg,  tokenizer_roberta)

---
# 1K Dataset

All four models trained on the 1 K Amazon review split, evaluated on `1k_test.csv` (100 samples, binary).

In [ ]:
CLASS_NAMES_BIN = ['Negative', 'Positive']

# ANN
y_true_ann_1k, y_pred_ann_1k = eval_ann(ann_model_1k, ann_vec_1k, ann_svd_1k, test_1k)
report('ANN â€” 1K', y_true_ann_1k, y_pred_ann_1k, CLASS_NAMES_BIN)

# BiLSTM
y_true_bilstm_1k, y_pred_bilstm_1k = eval_bilstm(bilstm_model_1k, bilstm_vocab_1k, bilstm_maxlen_1k, test_1k)
report('BiLSTM â€” 1K', y_true_bilstm_1k, y_pred_bilstm_1k, CLASS_NAMES_BIN)

# DistilBERT
_, y_true_distil_1k, y_pred_distil_1k = evaluate_tf(
    get_trainer(distilbert_1k), ds_1k_distil,
    label='DistilBERT â€” 1K', class_names=CLASS_NAMES_BIN,
)

# RoBERTa
_, y_true_roberta_1k, y_pred_roberta_1k = evaluate_tf(
    get_trainer(roberta_1k), ds_1k_roberta,
    label='RoBERTa â€” 1K', class_names=CLASS_NAMES_BIN,
)

In [ ]:
results_1k = [
    compute_scores('ANN',        y_true_ann_1k,    y_pred_ann_1k),
    compute_scores('BiLSTM',     y_true_bilstm_1k, y_pred_bilstm_1k),
    compute_scores('DistilBERT', y_true_distil_1k, y_pred_distil_1k),
    compute_scores('RoBERTa',    y_true_roberta_1k, y_pred_roberta_1k),
]
df_1k = pd.DataFrame(results_1k).set_index('Model')
df_1k.style.format('{:.4f}').highlight_max(axis=0, props='font-weight: bold; color: #1a7f3c')

In [ ]:
show_cm_grid([
    (y_true_ann_1k,     y_pred_ann_1k,     2, CLASS_NAMES_BIN, 'ANN 1K'),
    (y_true_bilstm_1k,  y_pred_bilstm_1k,  2, CLASS_NAMES_BIN, 'BiLSTM 1K'),
    (y_true_distil_1k,  y_pred_distil_1k,  2, CLASS_NAMES_BIN, 'DistilBERT 1K'),
    (y_true_roberta_1k, y_pred_roberta_1k, 2, CLASS_NAMES_BIN, 'RoBERTa 1K'),
])

---
# 25K Dataset

All four models trained on the 25 K Amazon review split, evaluated on `25k_test.csv` (2 500 samples, binary).

In [ ]:
# ANN
y_true_ann_25k, y_pred_ann_25k = eval_ann(ann_model_25k, ann_vec_25k, ann_svd_25k, test_25k)
report('ANN â€” 25K', y_true_ann_25k, y_pred_ann_25k, CLASS_NAMES_BIN)

# BiLSTM
y_true_bilstm_25k, y_pred_bilstm_25k = eval_bilstm(bilstm_model_25k, bilstm_vocab_25k, bilstm_maxlen_25k, test_25k)
report('BiLSTM â€” 25K', y_true_bilstm_25k, y_pred_bilstm_25k, CLASS_NAMES_BIN)

# DistilBERT
_, y_true_distil_25k, y_pred_distil_25k = evaluate_tf(
    get_trainer(distilbert_25k), ds_25k_distil,
    label='DistilBERT â€” 25K', class_names=CLASS_NAMES_BIN,
)

# RoBERTa
_, y_true_roberta_25k, y_pred_roberta_25k = evaluate_tf(
    get_trainer(roberta_25k), ds_25k_roberta,
    label='RoBERTa â€” 25K', class_names=CLASS_NAMES_BIN,
)

In [ ]:
results_25k = [
    compute_scores('ANN',        y_true_ann_25k,    y_pred_ann_25k),
    compute_scores('BiLSTM',     y_true_bilstm_25k, y_pred_bilstm_25k),
    compute_scores('DistilBERT', y_true_distil_25k, y_pred_distil_25k),
    compute_scores('RoBERTa',    y_true_roberta_25k, y_pred_roberta_25k),
]
df_25k = pd.DataFrame(results_25k).set_index('Model')
df_25k.style.format('{:.4f}').highlight_max(axis=0, props='font-weight: bold; color: #1a7f3c')

In [ ]:
show_cm_grid([
    (y_true_ann_25k,    y_pred_ann_25k,    2, CLASS_NAMES_BIN, 'ANN 25K'),
    (y_true_bilstm_25k, y_pred_bilstm_25k, 2, CLASS_NAMES_BIN, 'BiLSTM 25K'),
    (y_true_distil_25k, y_pred_distil_25k, 2, CLASS_NAMES_BIN, 'DistilBERT 25K'),
    (y_true_roberta_25k, y_pred_roberta_25k, 2, CLASS_NAMES_BIN, 'RoBERTa 25K'),
])

---
# Video Games Dataset

All four models trained on the Video Games split, evaluated on `vg_test.csv` (~256 K samples, 5-class star rating).

In [ ]:
CLASS_NAMES_VG = ['1-star', '2-star', '3-star', '4-star', '5-star']

# ANN
y_true_ann_vg, y_pred_ann_vg = eval_ann(ann_model_vg, ann_vec_vg, ann_svd_vg, test_vg)
report('ANN â€” VG', y_true_ann_vg, y_pred_ann_vg, CLASS_NAMES_VG)

# BiLSTM
y_true_bilstm_vg, y_pred_bilstm_vg = eval_bilstm(bilstm_model_vg, bilstm_vocab_vg, bilstm_maxlen_vg, test_vg)
report('BiLSTM â€” VG', y_true_bilstm_vg, y_pred_bilstm_vg, CLASS_NAMES_VG)

# DistilBERT
_, y_true_distil_vg, y_pred_distil_vg = evaluate_tf(
    get_trainer(distilbert_vg), ds_vg_distil,
    label='DistilBERT â€” VG', class_names=CLASS_NAMES_VG,
)

# RoBERTa
_, y_true_roberta_vg, y_pred_roberta_vg = evaluate_tf(
    get_trainer(roberta_vg), ds_vg_roberta,
    label='RoBERTa â€” VG', class_names=CLASS_NAMES_VG,
)

In [ ]:
results_vg = [
    compute_scores('ANN',        y_true_ann_vg,    y_pred_ann_vg),
    compute_scores('BiLSTM',     y_true_bilstm_vg, y_pred_bilstm_vg),
    compute_scores('DistilBERT', y_true_distil_vg, y_pred_distil_vg),
    compute_scores('RoBERTa',    y_true_roberta_vg, y_pred_roberta_vg),
]
df_vg = pd.DataFrame(results_vg).set_index('Model')
df_vg.style.format('{:.4f}').highlight_max(axis=0, props='font-weight: bold; color: #1a7f3c')

In [ ]:
show_cm_grid([
    (y_true_ann_vg,     y_pred_ann_vg,     5, CLASS_NAMES_VG, 'ANN VG',        True),
    (y_true_bilstm_vg,  y_pred_bilstm_vg,  5, CLASS_NAMES_VG, 'BiLSTM VG',     True),
    (y_true_distil_vg,  y_pred_distil_vg,  5, CLASS_NAMES_VG, 'DistilBERT VG', True),
    (y_true_roberta_vg, y_pred_roberta_vg, 5, CLASS_NAMES_VG, 'RoBERTa VG',    True),
])

---
# Overall Summary

All 12 modelâ€“dataset combinations in a single table, grouped by dataset for easy cross-model comparison.

In [ ]:
all_results = []
for name, y_true, y_pred, dataset in [
    ('ANN',        y_true_ann_1k,     y_pred_ann_1k,     '1K'),
    ('BiLSTM',     y_true_bilstm_1k,  y_pred_bilstm_1k,  '1K'),
    ('DistilBERT', y_true_distil_1k,  y_pred_distil_1k,  '1K'),
    ('RoBERTa',    y_true_roberta_1k, y_pred_roberta_1k, '1K'),
    ('ANN',        y_true_ann_25k,    y_pred_ann_25k,    '25K'),
    ('BiLSTM',     y_true_bilstm_25k, y_pred_bilstm_25k, '25K'),
    ('DistilBERT', y_true_distil_25k, y_pred_distil_25k, '25K'),
    ('RoBERTa',    y_true_roberta_25k,y_pred_roberta_25k,'25K'),
    ('ANN',        y_true_ann_vg,     y_pred_ann_vg,     'VG'),
    ('BiLSTM',     y_true_bilstm_vg,  y_pred_bilstm_vg,  'VG'),
    ('DistilBERT', y_true_distil_vg,  y_pred_distil_vg,  'VG'),
    ('RoBERTa',    y_true_roberta_vg, y_pred_roberta_vg, 'VG'),
]:
    row = compute_scores(name, y_true, y_pred)
    row['Dataset'] = dataset
    all_results.append(row)

df_all = (
    pd.DataFrame(all_results)
    .set_index(['Dataset', 'Model'])
)
df_all.style.format('{:.4f}').highlight_max(axis=0, props='font-weight: bold; color: #1a7f3c')